In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt, Production
from pydantic import BaseModel

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [2]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')


Load started on 04/06/2026 16:40:17
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/06/2026 16:40:17
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/06/2026 16:40:17
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/06/2026 16:40:17
Loading file Agents.Operation.ToolkitUtilities.cls as udl
Compiling class Agents.Operation.ToolkitUtilities
Compiling routine Agents.Operation.ToolkitUtilities.1
Load finished successfully.

Load started on 04/06/2026 16:40:17
Loading file Agents.Message.ToolRequest.cls as ud

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [3]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [4]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [5]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [6]:
bond_system = Prompt(name = 'Agent007', text = 'You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [7]:
bond_system = Prompt(name = 'Agent007', text = 'Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [8]:
Prompt('Agent007') == bond_system

True

In [9]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [10]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [11]:
Prompt('Agent007').delete()
try:
    prompt = Prompt("Agent007")
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [12]:
molly = Agent(name='Molly', model='gpt-5')
Production('AgentSpace', [molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 04/06/2026 16:40:19
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/06/2026 16:40:19
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/06/2026 16:40:19
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/06/2026 16:40:20
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/06/2026 16:40:20
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are solid summer hiking options around Boston (with quick notes on distance, vibe, and access):\n\n- Middlesex Fells Reservation (Medford/Stoneham/Winchester) – Skyline Trail 7–9 mi with rocky sections and Boston views; many shorter pond/woodland loops. MBTA: Orange Line to Oak Grove/Wellington + walk/ride.\n- Blue Hills Reservation (Milton/Quincy/Canton) – Skyline Traverse 9–10 mi; Great Blue Hill summit 2–3 mi; Ponkapoag Bog boardwalk. MBTA bus to Houghton’s Pond area. Mix of shade and exposed ledges.\n- Lynn Woods Reservation (Lynn) – 3–8 mi loops to Stone Tower/Dungeon Rock; rolling, rugged; partial shade. MBTA bus accessible.\n- Breakheart Reservation (Saugus) – 2 mi paved inner loop plus rugged side trails; swim beach; family-friendly.\n- Minute Man NHP: Battle Road Trail (Lexington/Concord) – ~5 mi one-way, flat and historic with shade breaks. Commuter Rail to Concord, then local transit/ride.\n- Walden Pond State Reservation (Concord) – 1.7 mi shoreline loop; connect to f

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [13]:
Agent('Molly') == molly

True

In [14]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 04/06/2026 16:40:51
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 04/06/2026 16:40:52
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/06/2026 16:40:52
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/06/2026 16:40:52
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/06/2026 16:40:52
Loading file Agents.Ope

In [15]:
Production('AgentSpace', [molly, alex]).start()


Load started on 04/06/2026 16:40:55
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/06/2026 16:40:55
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/06/2026 16:40:55
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/06/2026 16:40:55
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/06/2026 16:40:55
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

In [16]:
molly(message='Which tables do we have in IRIS in the Agents namespace?')

'{"text": "Tables in the Agents namespace:\\n- SQLUser.Agent\\n- SQLUser.AgentToolkit\\n- SQLUser.Chat\\n- SQLUser.Prompt\\n- SQLUser.TestModel\\n- SQLUser.Toolkit\\n- SQLUser.ToolUsage\\n- SQLUser.Usage", "reasoning": "Used the provided IRIS list_tables result for the Agents namespace to extract the table names."}'

In [17]:
molly(message='What is the weather today?', chat=context)

'{"text": "Today in Washington, DC: Cloudy, high 26, low 13.", "reasoning": "Used the successful Utilities.weather tool result for Washington, DC."}'

In [18]:
molly(message='Recommend some good food spots for lunch', chat='travel')

'{"text": "Here are solid lunch spots around DC:\\n\\nQuick-casual:\\n- RASA (Dupont, Navy Yard): Build-your-own Indian bowls; great veg options.\\n- Shouk (Mt. Vernon Sq., Georgetown): Plant-based Israeli street food.\\n- Little Sesame (Downtown, Golden Triangle): Hummus bowls and pita sandwiches.\\n- A Baked Joint (Mt. Vernon Triangle): Big sandwiches, salads, excellent bread.\\n- Falafel Inc (Georgetown): Fast, budget-friendly falafel.\\n- Arepa Zone (14th St, Union Market): Venezuelan arepas and teque\\u00f1os.\\n- Call Your Mother (multiple): Bagel sandwiches; expect a line.\\n\\nSit-down:\\n- Le Diplomate (Logan Circle): Classic French brasserie; reserve if you can.\\n- Old Ebbitt Grill (near the White House): DC institution; oysters, American fare.\\n- Zaytinya (Penn Quarter): Eastern Mediterranean small plates.\\n- Jaleo (Penn Quarter): Spanish tapas by Jos\\u00e9 Andr\\u00e9s.\\n- Daikaya (Chinatown): Sapporo-style ramen.\\n- The Salt Line (Navy Yard/Wharf): New England-style 

In [19]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 04/06/2026 16:41:43
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 04/06/2026 16:41:43
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "Sfoglina", "cuisine": "Italian"}, {"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "Centrolina", "cuisine": "Italian"}, {"name": "2Amys", "cuisine": "Italian"}, {"name": "Officina", "cuisine": "Italian"}, {"name": "Stellina Pizzeria", "cuisine": "Italian"}, {"name": "Rasika", "cuisine": "Indian"}, {"name": "Daikaya", "cuisine": "Japanese"}, {"name": "Sushi Capitol", "cuisine": "Japanese"}, {"name": "CHIKO", "cuisine": "Chinese-Korean"}, {"name": "Maketto", "cuisine": "Cambodian/Taiwanese"}, {"name": "Pho 14", "cuisine": "Vietnamese"}], "reasoning": "Curated Washington, DC spots matching Italian and Asian preferences; mix of reliable casual and sit-down options."}'

In [20]:
Chat('travel').usage()

'{"input_tokens": 12143, "output_tokens": 22011, "output_reasoning_tokens": 18496, "total_tokens": 34154}'

In [21]:
Production('AgentSpace').usage()

{'input_tokens': 14044,
 'output_tokens': 25855,
 'output_reasoning_tokens': 20864,
 'total_tokens': 39899}

In [22]:
molly.usage()

{'input_tokens': 19136,
 'output_tokens': 35992,
 'output_reasoning_tokens': 28416,
 'total_tokens': 55128}

In [23]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 14044,
 'output_tokens': 25855,
 'output_reasoning_tokens': 20864,
 'total_tokens': 39899}

In [24]:
Production('AgentSpace').delete()

Deleted production: User.AgentSpace

Deleting class Agents.REST.Dispatch.AgentSpaceCleaned up production-owned artifacts for: AgentSpace


In [25]:
Agent('Molly').delete()
try:
    molly('Hello')
except KeyError as e:
    print(e)


Deleting class Agents.Gateway.MollyService
Deleting class Agents.Process.Molly"No Agent found for 'Molly'"
